# 实战案例：基于多模态因子双线性池化（MFB）的视觉问答

**本notebook已从PaddlePaddle转换为PyTorch，默认在CPU上运行。**

## 视觉问答技术简介

视觉问答的关键是挖掘图像和问题之间的关联，并融合二者推理出答案。形式上，视觉问答模型需要融合图像和文本两个模态的输入信息，预测出文本模态形式的答案或答案对应的类别编号。

现有的视觉问答技术可以被细分为四类：一是基于特征融合的方法，即利用线性融合或者双线性融合方法综合图像和问题的信息；二是基于注意力的方法，即利用注意力机制将图像中的区域和问题进行多模态对齐和融合；三是基于视觉关系建模的方法，该方法显示的建模图像中区域间的关系，并利用多模态对齐技术建立这些关系和问题的联系，最终利用多模态融合技术综合关系和问题；四是基于模块网络的方法，其首先利用全连接层、卷积层、注意力层等单元预定义一系列模块，包括属性或实体查找模块、关注区域转换模块、组合模块、属性描述模块、计数模块等，然后将问题解析为可以和这些模块对应的部分，最后依据对问题的解析结果动态的组装预定义的模块。

需要说明的是，这四类方法并非完全独立，例如使用这四类方法构建视觉问答模型时，大多都会使用注意力机制进行多模态融合和对齐。

这里我们将具体介绍一个基于多模态因子双线性池化 (Multimodal Factorized Bilinear, MFB)和注意力的视觉问答模型（MFBVQA模型）。该模型首先利用注意力将问题和图像中的区域进行多模态对齐，其中注意力评分函数为MFB，然后再次利用MFB融合多模态对齐前后的问题表示，最终推理出答案。

下面，我们将按照读取数据、定义模型、定义损失函数、选择优化方法、选择评估指标和训练模型的顺序，依次介绍该模型的具体实现。

## 环境准备

安装必要的依赖并导入基础库。本项目默认使用CPU运行。

In [ ]:
!pip install torch numpy pillow matplotlib -q

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import numpy as np
import json
import os
import random
import re
from os.path import join as pjoin
from argparse import Namespace
from collections import defaultdict, Counter

device = torch.device('cpu')
print(f'使用设备: {device}')
print(f'PyTorch版本: {torch.__version__}')

## 读取数据

可以直接运行下面的命令，生成已经处理的数据，直接跳到定义数据集类章节。从下载数据集到定义数据集之前的小节代码的功能正是从原始数据生成处理过的数据的功能，我们默认不运行。

In [ ]:
!tar -zxf ./data/data245864/vqa_processed.tar.gz -C ./data
!mv ./data/vqa_processed ./data/vqa

### 下载数据集

我们使用的数据集为VQA v2([下载地址](https://visualqa.org/download.html))。该数据集中的图像来自于MS COCO，我们需要下载[bottom up attention模型](https://github.com/peteanderson80/bottom-up-attention)提供的图像区域表示。该文件解压后的tsv文件包含了MS COCO训练集和验证集中所有图片的36个检测框的视觉表示。本节的代码将tsv文件放在目录data/vqa/coco下。

对于问题和回答，我们需要下载四个文件：训练标注集（训练回答集）、验证标注集（训练回答集）、训练问题集和验证问题集。将这四个文件解压后，我们得到4个json格式的文件，并将其放在指定目录(本节的代码中将该目录设置为data/vqa)下的vqa2文件夹里。由于测试集的标注集未公开，因此这里仅下载训练集和验证集。下载的数据集包含443,757个训练问题和214,354个验证问题，每个问题对应10个人工标注的答案。

### 整理数据集

数据集下载完成后，我们需要对其进行处理，以适合之后构造的PyTorch数据集类读取。
对于图像，我们将每张图片的36个检测框表示存储为单个npy格式文件，并将文件路径记录在数据json文件中。为了后续的数据分析，我们还将检测框的位置信息也以npy格式存储。json文件中的路径仅存储文件名前缀，加上后缀'.npy'为图像特征，加上后缀'.box.npy'为检测框特征。

In [ ]:
import base64
import csv
import sys

csv.field_size_limit(sys.maxsize)

def resort_image_feature():
    image_feature_path = 'data/vqa/coco/trainval_resnet101_faster_rcnn_genome_36.tsv' 
    feature_folder = 'data/vqa/coco/image_box_features'
    if not os.path.exists(feature_folder):
        os.makedirs(feature_folder)
        
    FIELDNAMES = ['image_id', 'image_h', 'image_w', 'num_boxes', 'boxes', 'features']
    with open(image_feature_path, 'r') as tsv_in_file:
        reader = csv.DictReader(tsv_in_file, delimiter='\t', fieldnames=FIELDNAMES)
        for item in reader:
            item['num_boxes'] = int(item['num_boxes'])
            for field in ['boxes', 'features']:
                buf = base64.b64decode(item[field])
                temp = np.frombuffer(buf, dtype=np.float32)
                item[field] = temp.reshape((item['num_boxes'], -1))
            np.save(pjoin(feature_folder, item['image_id']+'.jpg.npy'), item['features'])
            np.save(pjoin(feature_folder, item['image_id']+'.jpg.box.npy'), item['boxes'])

# resort_image_feature()

对于回答，我们取出现频次最高的max_ans_count个回答，将任务转化为max_ans_count个类的分类任务，并将每个问题的多个回答转化为列表。

对于问题，我们首先构建词典，然后根据词典将问题转化为向量，并过滤掉所有回答都不在高频回答中的问题样本。

In [ ]:
from PIL import Image

def tokenize_mcb(s):
    """
    问题词元化（tokenization）函数
    """
    t_str = s.lower()
    for i in [r'\?',r'\!',r'\'',r'\"',r'\$',r'\:',r'\@',r'\(',r'\)',r'\,',r'\.',r'\;']:
        t_str = re.sub(i, '', t_str)
    for i in [r'\-',r'\/']:
        t_str = re.sub(i, ' ', t_str)
    q_list = re.sub(r'\?','',t_str.lower()).split(' ')
    q_list = list(filter(lambda x: len(x) > 0, q_list))
    return q_list

def tokenize_questions(questions):
    for item in questions:
        item['question_tokens'] = tokenize_mcb(item['question'])
    return questions

def annotations_in_top_answers(annotations, questions, ans_vocab):
    new_anno = []
    new_ques = []
    assert len(annotations) == len(questions)
    for anno, ques in zip(annotations, questions):
        if anno['multiple_choice_answer'] in ans_vocab:
            new_anno.append(anno)
            new_ques.append(ques)
    return new_anno, new_ques
    
def encode_questions(questions, vocab):
    for item in questions:
        item['question_idx'] = [vocab.get(w, vocab['<unk>']) for w in item['question_tokens']]
    return questions

def encode_answers(annotations, vocab):
    for item in annotations:
        item['answer_list'] = []
        answers = [a['answer'] for a in item['answers']]
        for ans in answers:
            if ans in vocab:
                item['answer_list'].append(vocab[ans])
    return annotations

def create_dataset(dataset='flickr8k',
                   max_ans_count=1000, 
                   min_word_count=10):
    """
    参数：
        dataset：数据集名称
        max_ans_count：取训练集中最高频的1000个答案
        min_word_count：仅考虑在训练集中问题文本里出现10次及以上的词
    输出：
        一个词典文件： vocab.json
        两个数据集文件： train_data.json、 val_data.json
    """
    dir_vqa2 = 'data/vqa/vqa2'
    dir_processed = os.path.join(dir_vqa2, 'processed')
    dir_ann = pjoin(dir_vqa2, 'raw', 'annotations')
    path_train_ann = pjoin(dir_ann, 'mscoco_train2014_annotations.json')
    path_train_ques = pjoin(dir_ann, 'OpenEnded_mscoco_train2014_questions.json')
    path_val_ann = pjoin(dir_ann, 'mscoco_val2014_annotations.json')
    path_val_ques = pjoin(dir_ann, 'OpenEnded_mscoco_val2014_questions.json')

    train_anno = json.load(open(path_train_ann))['annotations']
    train_ques = json.load(open(path_train_ques))['questions']
    val_anno = json.load(open(path_val_ann))['annotations']
    val_ques = json.load(open(path_val_ques))['questions']
    
    ans2ct = defaultdict(int)
    for item in train_anno:
        ans = item['multiple_choice_answer'] 
        ans2ct[ans] += 1
    ans_ct = sorted(ans2ct.items(), key=lambda item: item[1], reverse=True)
    ans_vocab = [ans_ct[i][0] for i in range(max_ans_count)] 
    ans_vocab = {a: i for i, a in enumerate(ans_vocab)}
    train_anno = encode_answers(train_anno, ans_vocab)
    val_anno = encode_answers(val_anno, ans_vocab)

    train_ques = tokenize_questions(train_ques)
    val_ques = tokenize_questions(val_ques)

    ques_vocab = Counter()
    for item in train_ques:
        ques_vocab.update(item['question_tokens'])
    ques_vocab = [w for w in ques_vocab.keys() if ques_vocab[w] > min_word_count]
    ques_vocab = {q: i for i, q in enumerate(ques_vocab)}
    ques_vocab['<unk>'] = len(ques_vocab)
    train_ques = encode_questions(train_ques, ques_vocab)
    val_ques = encode_questions(val_ques, ques_vocab)

    train_anno, train_ques = annotations_in_top_answers(
            train_anno, train_ques, ans_vocab)

    if not os.path.exists(dir_processed):
        os.makedirs(dir_processed)
    with open(pjoin(dir_processed, 'vocab.json'), 'w') as fw:
        json.dump({'ans_vocab': ans_vocab, 'ques_vocab': ques_vocab}, fw)
    with open(pjoin(dir_processed, 'train_data.json'), 'w') as fw:
        json.dump({'annotations': train_anno, 'questions': train_ques}, fw)
    with open(pjoin(dir_processed, 'val_data.json'), 'w') as fw:
        json.dump({'annotations': val_anno, 'questions': val_ques}, fw)
    
# create_dataset()

在调用该函数生成需要的格式的数据集文件之后，我们可以展示其中一条数据，简单验证下数据的格式是否和我们预想的一致。

In [ ]:
%matplotlib inline
from matplotlib import pyplot as plt

data_dir = './data/vqa/vqa2/'
dir_processed = pjoin(data_dir, 'processed')
rcnn_dir = './data/vqa/coco/image_box_features/'
image_dir = './data/vqa/coco/raw/val2014/'

vocab = json.load(open(pjoin(dir_processed, 'vocab.json'), 'r'))
dataset = json.load(open(pjoin(dir_processed, 'val_data.json'), 'r'))

idx2ans = {i: a for a, i in vocab['ans_vocab'].items()}
idx2ques = {i: q for q, i in vocab['ques_vocab'].items()}

idx = 10000
question = dataset['questions'][idx]
q_text = ' '.join([idx2ques[token] for token in question['question_idx']])
annotation = dataset['annotations'][idx]
a_text = '/'.join([idx2ans[token] for token in annotation['answer_list']])

image_name = 'COCO_val2014_%012d.jpg' % (question['image_id'])
content_img = Image.open(pjoin(image_dir, image_name))
fig = plt.imshow(content_img)
feats = np.load(pjoin(rcnn_dir, '{}.jpg.box.npy').format(question['image_id']))
for i in range(feats.shape[0]):
    bbox = feats[i, :]
    color = 'red' if i <= 3 else 'blue'
    fig.axes.add_patch(plt.Rectangle(
        xy=(bbox[0], bbox[1]), width=bbox[2]-bbox[0], height=bbox[3]-bbox[1],
        fill=False, edgecolor=color, linewidth=1))

### 定义数据集类

在准备好的数据集的基础上，我们需要进一步定义PyTorch Dataset类，以使用PyTorch DataLoader类按批次产生数据。

在PyTorch中定义数据集类非常简单，仅需要继承`torch.utils.data.Dataset`类，并实现`__getitem__`和`__len__`两个函数即可。

> **Paddle → PyTorch 转换要点：**
> - `paddle.io.Dataset` → `torch.utils.data.Dataset`
> - `paddle.to_tensor()` → `torch.tensor()`
> - `paddle.zeros()` → `torch.zeros()`
> - `DataLoader` 的 `collate_fn` 用法相同

In [ ]:
def collate_fn(batch):
    """对一个批次的数据进行预处理"""
    max_question_length = max([len(item['question']) for item in batch])
    batch_size = len(batch)
    imgs = torch.zeros((batch_size, batch[0]['image_feat'].shape[0], batch[0]['image_feat'].shape[1]))
    ques = torch.zeros((batch_size, max_question_length), dtype=torch.long)
    ans = torch.zeros((batch_size, 1000))
    lens = torch.zeros(batch_size, dtype=torch.long)
    for i, item in enumerate(batch):
        imgs[i] = torch.from_numpy(item['image_feat'])
        ques[i, :item['question'].shape[0]] = item['question']
        for answer in item['answers']:
            ans[i, answer] += 1
        lens[i] = item['length']
    return (imgs, ques, ans, lens)


class VQA2Dataset(Dataset):

    def __init__(self,
            data_dir='./data/vqa/vqa2/',
            rcnn_dir='./data/vqa/coco/image_box_features/',
            split='train',
            samplingans=True):
        """
        参数：
            samplingans: 决定返回的回答数据。
                取值为True，则从回答列表中按照回答出现的概率采样一个回答；
                取值为False，则为回答列表。
        """
        super(VQA2Dataset, self).__init__()
        self.rcnn_dir = rcnn_dir
        self.samplingans = samplingans
        self.split = split
        dir_processed = pjoin(data_dir, 'processed')
        if split == 'train':
            self.dataset = json.load(open(pjoin(dir_processed, 'train_data.json'), 'r'))
        elif split == 'val':
            self.dataset = json.load(open(pjoin(dir_processed, 'val_data.json'), 'r'))
        self.dataset_size = len(self.dataset['questions'])

    def __getitem__(self, index):
        item = {}
        item['index'] = index

        question = self.dataset['questions'][index]
        item['question'] = torch.tensor(question['question_idx'], dtype=torch.long)
        item['length'] = torch.tensor([len(question['question_idx'])], dtype=torch.long)
        item['image_feat'] = np.load(pjoin(self.rcnn_dir, '{}.jpg.npy'.format(question['image_id'])))

        annotation = self.dataset['annotations'][index]
        if 'train' in self.split and self.samplingans:
            item['answers'] = [random.choice(annotation['answer_list'])]
        else:
            item['answers'] = annotation['answer_list']
        return item

    def __len__(self):
        return self.dataset_size

### 批量读取数据

利用刚才构造的数据集类，借助DataLoader类构建能够按批次产生训练、验证和测试数据的对象。

In [ ]:
def mktrainval(data_dir, image_feat_dir, batch_size, workers=0):
    train_set = VQA2Dataset(data_dir, image_feat_dir, split='train', samplingans=True)
    valid_set = VQA2Dataset(data_dir, image_feat_dir, split='val', samplingans=False)

    train_loader = DataLoader(
                        train_set, batch_size=batch_size, 
                        shuffle=True, num_workers=workers, 
                        collate_fn=collate_fn)
    valid_loader = DataLoader(
                        valid_set, batch_size=batch_size, 
                        shuffle=False, num_workers=workers, 
                        drop_last=False, collate_fn=collate_fn)

    return train_loader, valid_loader

## 定义模型

MFBVQA模型主要包含两个模块：注意力跨模态对齐模块和双线性融合模块。

注意力跨模态对齐模块使用问题表示作为查询，图像的局部表示作为键和值，获得问题和图像对齐的表示。形式上，该表示为图像局部表示的加权求和的结果，权重则代表了图像区域和该问题的关联程度。需要注意的是，这里使用的是多头注意力，且注意力评分函数中计算查询和键的关联时，使用了MFB融合操作。

双线性融合模块使用MFB融合问题对齐前后的表示，获得最终的融合表示。

下面我们将首先实现MFB融合操作，然后实现基于MFB融合的注意力跨模态对齐，最后借助注意力跨模态对齐和MFB融合实现MFBVQA模型。

> **Paddle → PyTorch 转换要点：**
> - `nn.Layer` → `nn.Module`
> - `nn.LayerList` → `nn.ModuleList`
> - `paddle.repeat_interleave()` → `torch.repeat_interleave()`（用法相同）
> - `paddle.split(x, n, axis=1)` → `torch.split(x, 1, dim=1)`（注意：Paddle的split第二个参数是份数，PyTorch是每份大小）
> - `paddle.transpose(x, (0,2,1))` → `x.permute(0, 2, 1)`
> - `paddle.bmm()` → `torch.bmm()`
> - `nn.Softmax(axis=1)` → `nn.Softmax(dim=1)`

### MFB融合

下面展示MFB融合操作的实现。该函数既支持两个向量融合，也支持两组向量的融合。

In [ ]:
class MFBFusion(nn.Module):
    def __init__(self, input_dim1, input_dim2, hidden_dim, R):
        '''
        参数：
            input_dim1: 第一个待融合表示的维度
            input_dim2: 第二个待融合表示的维度
            hidden_dim: 融合后的表示的维度
            R: MFB所使用的低秩矩阵的数量
        '''
        super(MFBFusion, self).__init__()
        self.input_dim1 = input_dim1
        self.input_dim2 = input_dim2
        self.hidden_dim = hidden_dim
        self.R = R
        self.linear1 = nn.Linear(input_dim1, hidden_dim * R)
        self.linear2 = nn.Linear(input_dim2, hidden_dim * R)

    def forward(self, inputs1, inputs2):
        '''
        参数：
            inputs1: (batch_size, input_dim1) 或 (batch_size, num_region, input_dim1)
            inputs2: (batch_size, input_dim2) 或 (batch_size, num_region, input_dim2)
        '''
        num_region = 1
        if inputs1.dim() == 3:
            num_region = inputs1.shape[1]
        h1 = self.linear1(inputs1)
        h2 = self.linear2(inputs2)
        z = h1 * h2
        z = z.reshape((z.shape[0], num_region, self.hidden_dim, self.R))
        z = z.sum(3).squeeze(1)
        return z

### 注意力跨模态对齐模块

下面展示了多头交叉注意力的实现。其中注意力得分 $\alpha$ 是使用MFB操作融合查询和键的结果。

In [ ]:
class MultiHeadATTN(nn.Module):
    def __init__(self, query_dim, kv_dim, mfb_input_dim, mfb_hidden_dim, num_head, att_dim):
        """
        参数：
            query_dim：问题表示（查询）的维度
            kv_dim：图像区域表示（键和值）的维度
            mfb_input_dim：融合操作的输入的维度
            mfb_hidden_dim：融合操作的输出的维度
            num_head：多头交叉注意力的头数
            att_dim：多头交叉注意力的输出表示（对齐后的表示）维度
        """
        super(MultiHeadATTN, self).__init__()
        assert att_dim % num_head == 0
        self.num_head = num_head
        self.att_dim = att_dim

        self.attn_w_1_q = nn.Sequential(
                            nn.Dropout(0.5),
                            nn.Linear(query_dim, mfb_input_dim),
                            nn.ReLU()
                          )
        self.attn_w_1_k = nn.Sequential(
                            nn.Dropout(0.5),
                            nn.Linear(kv_dim, mfb_input_dim),
                            nn.ReLU()
                          )
        self.attn_score_fusion = MFBFusion(mfb_input_dim, mfb_input_dim, mfb_hidden_dim, 1)
        self.attn_score_mapping = nn.Sequential(
                            nn.Dropout(0.5),
                            nn.Linear(mfb_hidden_dim, num_head)
                          )
        self.softmax = nn.Softmax(dim=1)
        # 对齐后的表示计算流程
        self.align_q = nn.ModuleList([nn.Sequential(
                            nn.Dropout(0.5),
                            nn.Linear(kv_dim, int(att_dim / num_head)),
                            nn.Tanh()
                       ) for _ in range(num_head)])

    def forward(self, query, key_value):
        """
        参数：
          query: (batch_size, q_dim)
          key_value: (batch_size, num_region, kv_dim)
        """
        #（1）使用全连接层将Q、K、V转化为向量
        num_region = key_value.shape[1]
        # -> (batch_size, num_region, mfb_input_dim)
        q = torch.repeat_interleave(self.attn_w_1_q(query).unsqueeze(1), num_region, dim=1)
        # -> (batch_size, num_region, mfb_input_dim)
        k = self.attn_w_1_k(key_value)
        #（2）计算query和key的相关性，实现注意力评分函数
        # -> (batch_size, num_region, num_head)
        alphas = self.attn_score_fusion(q, k)
        alphas = self.attn_score_mapping(alphas)
        #（3）归一化相关性分数
        # -> (batch_size, num_region, num_head)
        alphas = self.softmax(alphas)
        #（4）计算输出
        # (batch_size, num_region, num_head) (batch_size, num_region, key_value_dim)
        # -> (batch_size, num_head, key_value_dim)
        output = torch.bmm(alphas.permute(0, 2, 1), key_value)
        # 最终再对每个头的输出进行一次转换，并拼接所有头的转换结果作为注意力输出
        # torch.split: 第二个参数是每份大小（=1），而Paddle中是份数
        list_v = [e.squeeze(1) for e in torch.split(output, 1, dim=1)]
        alpha = torch.split(alphas, 1, dim=2)
        align_feat = torch.cat([self.align_q[head_id](x_v) for head_id, x_v in enumerate(list_v)], dim=1)
        return align_feat, alpha

### 模型

利用上述MFB融合操作和多头自注意力模块的实现，我们可以轻松的实现MFBVQA模型。模型的输入是图像的区域表示和问题。对于问题的表示，模型使用GRU编码器获取问题的整体表示。

> **Paddle → PyTorch GRU 转换要点：**
> - Paddle的`nn.GRU`支持直接传入`sequence_length`参数来处理变长序列
> - PyTorch的`nn.GRU`需要使用`pack_padded_sequence`和`pad_packed_sequence`来处理变长序列
> - Paddle的GRU hidden输出形状为`(num_layers * num_directions, batch, hidden_size)`，PyTorch相同
> - `nn.initializer.Uniform` → `nn.init.uniform_`

In [ ]:
class MFBVQAModel(nn.Module):
    def __init__(self, vocab_words, question_dim, image_dim, 
                       attn_mfb_input_dim, attn_mfb_hidden_dim, 
                       attn_num_head, attn_output_dim, 
                       fusion_q_feature_dim, fusion_mfb_hidden_dim,
                       num_classes):
        super(MFBVQAModel, self).__init__()

        # 文本表示提取器
        self.embed = nn.Embedding(len(vocab_words), 300)
        nn.init.uniform_(self.embed.weight, -0.1, 0.1)
        self.text_encoder = nn.GRU(300, question_dim, num_layers=2, batch_first=True)
        # 多头注意力
        self.attn = MultiHeadATTN(question_dim, image_dim, 
                                  attn_mfb_input_dim, attn_mfb_hidden_dim, 
                                  attn_num_head, attn_output_dim)
        # 问题的对齐表示到融合表示空间的映射函数
        self.q_feature_linear = nn.Sequential(
                                    nn.Dropout(0.5),
                                    nn.Linear(question_dim, fusion_q_feature_dim),
                                    nn.ReLU()
                                )
        # MFB融合图文表示类
        self.fusion = MFBFusion(attn_output_dim, fusion_q_feature_dim, fusion_mfb_hidden_dim, 2)
        # 分类器
        self.classifier_linear = nn.Sequential(
                                    nn.Dropout(0.5),
                                    nn.Linear(fusion_mfb_hidden_dim, num_classes)
                                )
        self.question_dim = question_dim

    def forward(self, imgs, quests, lengths):
        # 初始输入
        v_feature = imgs.reshape((-1, 36, 2048))
        x = self.embed(quests)
        
        # PyTorch GRU 使用 pack_padded_sequence 处理变长序列
        # 需要先按长度降序排列，再用 pack/unpack
        lengths_cpu = lengths.cpu()
        sorted_lengths, sorted_idx = torch.sort(lengths_cpu, descending=True)
        sorted_x = x[sorted_idx]
        
        # 确保长度至少为1，避免pack时出错
        sorted_lengths = sorted_lengths.clamp(min=1)
        
        packed = nn.utils.rnn.pack_padded_sequence(sorted_x, sorted_lengths.tolist(), batch_first=True)
        _, hidden = self.text_encoder(packed)
        # hidden: (num_layers, batch_size, hidden_size)
        
        # 恢复原始顺序
        _, unsorted_idx = torch.sort(sorted_idx)
        hidden = hidden[:, unsorted_idx, :]
        
        # 最后一层的隐藏状态
        q_feature = F.normalize(hidden[-1], dim=1)
        # 利用注意力获得问题的对齐表示
        align_q_feature, _ = self.attn(q_feature, v_feature)
        # 对原始文本表示进行变换
        original_q_feature = self.q_feature_linear(q_feature)
        # 融合对齐前后的问题的表示
        x = self.fusion(align_q_feature, original_q_feature)
        # 分类
        x = self.classifier_linear(x)
        return x

## 定义损失函数

模型的损失函数为KL散度损失，同时兼容回答为单一值和列表两种情形。

> **Paddle → PyTorch 转换要点：**
> - `nn.KLDivLoss(reduction='batchmean')` 用法相同
> - `nn.functional.log_softmax(input)` → `F.log_softmax(input, dim=-1)`（PyTorch要求显式指定dim）

In [ ]:
class KLLoss(nn.Module):
    def __init__(self):
        super(KLLoss, self).__init__()
        self.loss = nn.KLDivLoss(reduction='batchmean')

    def forward(self, input, target):
        return self.loss(F.log_softmax(input, dim=-1), target)

## 选择优化方法

我们选用Adam优化算法来更新模型参数，学习速率采用指数衰减方法。

> **Paddle → PyTorch 转换要点：**
> - `paddle.optimizer.lr.ExponentialDecay` → `torch.optim.lr_scheduler.ExponentialLR`
> - `paddle.optimizer.Adam` → `torch.optim.Adam`
> - PyTorch中优化器和学习率调度器是分开的，需要分别创建

In [ ]:
def get_optimizer(model, config):
    """学习速率指数衰减"""
    optimizer = torch.optim.Adam(model.parameters(), lr=config.learning_rate)
    # gamma为每个epoch的衰减因子，Paddle中gamma=0.5^(1/50000)是per-step的
    # 这里保持一致，在训练循环中每步调用scheduler.step()
    scheduler = torch.optim.lr_scheduler.ExponentialLR(optimizer, gamma=0.5 ** (1 / 50000))
    return optimizer, scheduler

## 评估指标

这里实现了VQAv2数据集中最常用的评估指标——回答准确率。具体而言，如果模型给出的回答在人工标注的10个回答中出现了3次及以上，则该回答的准确率为1，出现两次和一次的准确率分别为2/3和1/3。

> **Paddle → PyTorch 转换要点：**
> - `paddle.arange()` → `torch.arange()`
> - `output.argmax(axis=1)` → `output.argmax(dim=1)`
> - `model.eval()` / `model.train()` 用法相同
> - 使用 `torch.no_grad()` 减少评估时的内存占用

In [ ]:
def evaluate(data_loader, model):
    model.eval()
    accs = []
    with torch.no_grad():
        for i, (imgs, questions, answers, lengths) in enumerate(data_loader):
            imgs = imgs.to(device)
            questions = questions.to(device)
            answers = answers.to(device)
            lengths = lengths.to(device)
            
            output = model(imgs, questions, lengths)
            hit_cts = answers[torch.arange(output.shape[0]), output.argmax(dim=1)]
            for hit_ct in hit_cts:
                accs.append(min(1, hit_ct.item() / 3.0))
    model.train()
    return float(sum(accs)) / len(accs)

## 训练模型

训练模型过程可以分为读取数据、前馈计算、计算损失、更新参数、选择模型五个步骤。

> **Paddle → PyTorch 训练循环转换要点：**
> - `optimizer.clear_grad()` → `optimizer.zero_grad()`
> - `nn.utils.clip_grad_norm_()` 用法相同
> - `paddle.save()` → `torch.save()`
> - PyTorch中需要额外调用`scheduler.step()`来更新学习率
> - 使用`.to(device)`将数据送到指定设备（CPU）

In [ ]:
# 设置模型超参数和辅助变量
config = Namespace(
    question_dim=2400, 
    image_dim=2048, 
    attn_mfb_input_dim=310, 
    attn_mfb_hidden_dim=510,
    attn_num_head=2, 
    attn_output_dim=620,
    fusion_q_feature_dim=310,
    fusion_mfb_hidden_dim=510,
    num_ans=1000,
    batch_size=128,
    learning_rate=0.0001,
    margin=0.2,
    num_epochs=45,
    grad_clip=0.25,
    evaluate_step=360,
    checkpoint=None,
    best_checkpoint='./model/mfb/best_vqa2.ckpt',
    last_checkpoint='./model/mfb/last_vqa2.ckpt'
)

# 数据
data_dir = './data/vqa/vqa2/'
dir_processed = pjoin(data_dir, 'processed')

train_loader, valid_loader = mktrainval(data_dir, 
               './data/vqa/coco/image_box_features/', 
               config.batch_size, 
               workers=0)

# 模型
vocab = json.load(open(pjoin(dir_processed, 'vocab.json'), 'r'))  

# 随机初始化 或 载入已训练的模型
start_epoch = 0
checkpoint = config.checkpoint
if checkpoint is None:
    model = MFBVQAModel(vocab['ques_vocab'], 
                        config.question_dim, 
                        config.image_dim, 
                        config.attn_mfb_input_dim, 
                        config.attn_mfb_hidden_dim, 
                        config.attn_num_head, 
                        config.attn_output_dim, 
                        config.fusion_q_feature_dim, 
                        config.fusion_mfb_hidden_dim,
                        config.num_ans)
else:
    checkpoint_data = torch.load(checkpoint, map_location=device)
    start_epoch = checkpoint_data['epoch'] + 1
    model = MFBVQAModel(vocab['ques_vocab'], 
                        config.question_dim, 
                        config.image_dim, 
                        config.attn_mfb_input_dim, 
                        config.attn_mfb_hidden_dim, 
                        config.attn_num_head, 
                        config.attn_output_dim, 
                        config.fusion_q_feature_dim, 
                        config.fusion_mfb_hidden_dim,
                        config.num_ans)
    model.load_state_dict(checkpoint_data['model'])

model = model.to(device)

# 优化器和学习率调度器
optimizer, scheduler = get_optimizer(model, config)

# 开启训练模式
model.train()

# 损失函数
loss_fn = KLLoss()

# 创建模型保存目录
os.makedirs(os.path.dirname(config.best_checkpoint), exist_ok=True)

best_res = 0
print("开始训练")
fw = open('log.txt', 'w')
for epoch in range(start_epoch, config.num_epochs):
    for i, (imgs, questions, answers, lengths) in enumerate(train_loader):
        # 将数据移至设备
        imgs = imgs.to(device)
        questions = questions.to(device)
        answers = answers.to(device)
        lengths = lengths.to(device)
        
        optimizer.zero_grad()
        
        # 2. 前馈计算
        output = model(imgs, questions, lengths)
        # 3. 计算损失
        loss = loss_fn(output, answers)
        loss.backward()
        
        # 梯度截断
        if config.grad_clip > 0:
            nn.utils.clip_grad_norm_(model.parameters(), config.grad_clip)

        # 4. 更新参数
        optimizer.step()
        scheduler.step()

        state = {
                'epoch': epoch,
                'step': i,
                'model': model.state_dict(),
                'optimizer': optimizer.state_dict()
                }
        
        if (i + 1) % config.evaluate_step == 0:
            acc = evaluate(valid_loader, model)
            # 5. 选择模型
            if best_res < acc:
                best_res = acc
                torch.save(state, config.best_checkpoint)
            torch.save(state, config.last_checkpoint)
            log_msg = 'epoch: %d, step: %d, loss: %.2f, ACC: %.3f' % (
                epoch, i + 1, loss.item(), acc)
            print(log_msg)
            fw.write(log_msg + '\n')
fw.close()

## Paddle → PyTorch 转换速查表

| Paddle | PyTorch | 说明 |
|--------|---------|------|
| `nn.Layer` | `nn.Module` | 模型基类 |
| `nn.LayerList` | `nn.ModuleList` | 模块列表（确保子模块被正确注册） |
| `paddle.io.Dataset` | `torch.utils.data.Dataset` | 数据集基类 |
| `paddle.io.DataLoader` | `torch.utils.data.DataLoader` | 数据加载器 |
| `paddle.to_tensor(x, dtype='int64')` | `torch.tensor(x, dtype=torch.long)` | 创建张量 |
| `paddle.zeros(shape, dtype='int64')` | `torch.zeros(shape, dtype=torch.long)` | 零张量 |
| `paddle.repeat_interleave(x, n, axis)` | `torch.repeat_interleave(x, n, dim=axis)` | 按元素重复，`axis` → `dim` |
| `paddle.split(x, num_splits, axis)` | `torch.split(x, split_size, dim)` | 分割：Paddle传份数，PyTorch传每份大小 |
| `paddle.transpose(x, perm)` | `x.permute(*perm)` | 维度变换 |
| `paddle.bmm(a, b)` | `torch.bmm(a, b)` | 批矩阵乘法 |
| `paddle.concat(tensors, axis)` | `torch.cat(tensors, dim=axis)` | 拼接张量 |
| `nn.Softmax(axis=1)` | `nn.Softmax(dim=1)` | Softmax，`axis` → `dim` |
| `nn.functional.normalize(x)` | `F.normalize(x, dim=1)` | L2归一化（PyTorch需指定dim） |
| `nn.functional.log_softmax(x)` | `F.log_softmax(x, dim=-1)` | Log softmax（PyTorch需指定dim） |
| `nn.GRU(input, hidden, layers)` + `sequence_length` | `nn.GRU` + `pack_padded_sequence` | 变长序列处理 |
| `nn.initializer.Uniform(low, high)` | `nn.init.uniform_(tensor, a, b)` | 均匀分布初始化 |
| `optimizer.clear_grad()` | `optimizer.zero_grad()` | 梯度清零 |
| `paddle.optimizer.lr.ExponentialDecay` | `torch.optim.lr_scheduler.ExponentialLR` | 指数衰减学习率 |
| `paddle.save(state, path)` | `torch.save(state, path)` | 保存模型 |
| `paddle.load(path)` | `torch.load(path, map_location=device)` | 加载模型 |
| `model.train()` / `model.eval()` | `model.train()` / `model.eval()` | 训练/评估模式切换 |
| 无 | `torch.no_grad()` | 评估时关闭梯度计算以节省内存 |